# Import Library

In [1]:
# ==========================================
# CELL 1: IMPORT & INISIALISASI
# ==========================================
import pandas as pd
import numpy as np
import torch
import re
from tqdm import tqdm

# NLP Augmentation
import nlpaug.augmenter.word as naw
import nltk
from deep_translator import GoogleTranslator

# Sastrawi untuk TF-IDF
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Scikit-Learn
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# DagsHub & MLflow
import dagshub
import mlflow
import mlflow.sklearn

# 1. Cek GPU (CUDA)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Menggunakan perangkat: {device}")

# 2. Inisialisasi Sastrawi
print("⏳ Menyiapkan Sastrawi Stemmer & Stopword...")
stemmer = StemmerFactory().create_stemmer()
stopword = StopWordRemoverFactory().create_stop_word_remover()

print("✅ Cell 1 Selesai: Semua library siap tempur!")

KeyboardInterrupt: 

# Import Data

In [ ]:
# ==========================================
# CELL 2: LOAD DATA, SPLIT, & AUGMENTASI (SUPER FAST MULTITHREADING)
# ==========================================
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
from deep_translator import GoogleTranslator
import concurrent.futures

print("📥 Memuat Golden Dataset Exigen...")

# 1. LOAD DATASET MASTER (Hanya 1 file yang sudah bersih!)
path_master = "../../../data/dataset_tiket_master_bersih.csv"
# Ganti baris 14 Anda menjadi seperti ini:
df_master = pd.read_csv(path_master, sep='|' , on_bad_lines='skip')
print(f"✅ Master Dataset Terbaca : {len(df_master)} baris\n" + "-" * 50)

# 2. FILTERING RE-AUDIT (Memastikan target valid)
kolom_target = ['tipe_aset', 'lokasi_gedung', 'lokasi_lantai', 'lokasi_zona', 'kategori_aset', 'severity']

# Pastikan tidak ada missing values di kolom target maupun input teks
df_master = df_master.dropna(subset=kolom_target + ['teks_keluhan_awam'])

# Pastikan severity hanya berisi 4 kelas utama
df_master = df_master[df_master['severity'].isin(['Ringan', 'Sedang', 'Berat', 'Fatal'])]

# 3. PEMISAHAN X (Input) dan Y (Target), LALU SPLIT
X_raw = df_master['teks_keluhan_awam'].astype(str)
Y = df_master[kolom_target]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(X_raw, Y, test_size=0.2, random_state=42)

print(f"🗂️ Data Latih Asli (X_train_raw) : {len(X_train_raw)} baris")
print(f"🗂️ Data Uji Eksperimen (X_test)  : {len(X_test_raw)} baris\n" + "-" * 50)


# ==========================================
# 4. AUGMENTASI DENGAN MULTITHREADING (PARALEL)
# ==========================================
print("⏳ Memulai Augmentasi Data Training (Mode Cepat 10 Pekerja/Thread)...")

def proses_satu_baris(args):
    """Fungsi pekerja yang akan dijalankan bersamaan oleh banyak thread"""
    teks, label_asli = args
    severity_kelas = label_asli[5] # Index 5 adalah severity
    
    n_copies = 0
    if severity_kelas == 'Fatal': n_copies = 6
    elif severity_kelas == 'Berat': n_copies = 3
    elif severity_kelas in ['Ringan', 'Sedang']: n_copies = 2
        
    hasil_sementara = [(teks, label_asli)] # Simpan teks aslinya dulu
    
    if n_copies > 0:
        try:
            # Terjemahkan ID -> EN -> ID
            en_text = GoogleTranslator(source='id', target='en').translate(teks)
            id_text = GoogleTranslator(source='en', target='id').translate(en_text)
            teks_alternatif = id_text
        except Exception:
            # Jika Google menolak koneksi (rate limit), fallback pakai teks asli
            teks_alternatif = teks 
            
        for _ in range(n_copies):
            hasil_sementara.append((teks_alternatif, label_asli))
            
    return hasil_sementara

# Siapkan antrean tugas
antrean_tugas = [(teks, y_train.iloc[i].values) for i, teks in enumerate(X_train_raw)]

X_train_aug_list = []
Y_train_aug_list = []

# Gunakan 10 pekerja paralel
with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
    for hasil_baris in tqdm(executor.map(proses_satu_baris, antrean_tugas), total=len(antrean_tugas), desc="Translating (Paralel)"):
        for res_teks, res_label in hasil_baris:
            X_train_aug_list.append(res_teks)
            Y_train_aug_list.append(res_label)

X_train_aug = pd.Series(X_train_aug_list)
y_train_aug = pd.DataFrame(Y_train_aug_list, columns=kolom_target)

print(f"\n✅ 'Back-Translation SMOTE' Super Cepat Selesai!")
print(f"📈 Data Training naik dari {len(X_train_raw)} menjadi {len(X_train_aug)} baris.")

📥 Memuat Golden Dataset Exigen...
✅ Master Dataset Terbaca : 1887 baris
--------------------------------------------------
🗂️ Data Latih Asli (X_train_raw) : 1509 baris
🗂️ Data Uji Eksperimen (X_test)  : 378 baris
--------------------------------------------------
⏳ Memulai Augmentasi Data Training (Mode Cepat 10 Pekerja/Thread)...


Translating (Paralel):   0%|          | 0/1509 [00:00<?, ?it/s]


✅ 'Back-Translation SMOTE' Super Cepat Selesai!
📈 Data Training naik dari 1509 menjadi 6096 baris.


In [ ]:
# ==========================================
# CELL 2.1: DIAGNOSTIK LABEL AUGMENTASI
# ==========================================
print("📊 1. Distribusi Kelas SEVERITY (SEBELUM Augmentasi):")
print(y_train['severity'].value_counts())
print("-" * 50)

print("📈 2. Distribusi Kelas SEVERITY (SESUDAH 'NLP SMOTE'):")
# Memastikan target kita benar-benar jadi seimbang
print(y_train_aug['severity'].value_counts())
print("-" * 50)

print("👀 3. Pengecekan Silang (Cross-Check) Teks vs Label:")
# Gabungkan X dan Y augmentasi untuk kita intip
df_cek_aug = pd.DataFrame({
    'teks_augmentasi': X_train_aug,
    'label_severity': y_train_aug['severity'],
    'label_kategori': y_train_aug['kategori_aset'] # Cek juga apakah kolomnya tidak tertukar
})

# Mari kita intip 10 baris pertama khusus untuk kelas 'Fatal'
print("\n>>> CONTOH TEKS KELAS FATAL HASIL AUGMENTASI:")
display(df_cek_aug[df_cek_aug['label_severity'] == 'Fatal'].head(10))

# Mari kita intip 5 baris pertama khusus untuk kelas 'Ringan'
print("\n>>> CONTOH TEKS KELAS RINGAN HASIL AUGMENTASI:")
display(df_cek_aug[df_cek_aug['label_severity'] == 'Ringan'].head(5))

📊 1. Distribusi Kelas SEVERITY (SEBELUM Augmentasi):
severity
Sedang    456
Berat     389
Ringan    369
Fatal     295
Name: count, dtype: int64
--------------------------------------------------
📈 2. Distribusi Kelas SEVERITY (SESUDAH 'NLP SMOTE'):
severity
Fatal     2065
Berat     1556
Sedang    1368
Ringan    1107
Name: count, dtype: int64
--------------------------------------------------
👀 3. Pengecekan Silang (Cross-Check) Teks vs Label:

>>> CONTOH TEKS KELAS FATAL HASIL AUGMENTASI:


,teks_augmentasi,label_severity,label_kategori
3,Mohon bantuan untuk perbaikan Fingerprint di G...,Fatal,Security Sistem
4,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
5,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
6,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
7,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
8,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
9,Mohon bantuannya untuk perbaikan sidik jari di...,Fatal,Security Sistem
10,Lampu UV di Gedung Gedung Servis lantai 3 zona...,Fatal,Electrical
11,Lampu UV di Service Building lantai 3 zona Uta...,Fatal,Electrical
12,Lampu UV di Service Building lantai 3 zona Uta...,Fatal,Electrical



>>> CONTOH TEKS KELAS RINGAN HASIL AUGMENTASI:


,teks_augmentasi,label_severity,label_kategori
27,"Halo, Lampu LED Strip di Gedung Gedung Parkir ...",Ringan,Electrical
28,Halo lampu LED strip di gedung parkir lantai 4...,Ringan,Electrical
29,Halo lampu LED strip di gedung parkir lantai 4...,Ringan,Electrical
80,Mohon periksa Lampu Downlight di Gedung Utama ...,Ringan,Electrical
81,Mohon dilakukan pengecekan Downlight pada Gedu...,Ringan,Electrical


# Cleaning & Preprocessing

In [ ]:
# ==========================================
# CELL 3: PREPROCESSING (SLANG KAMUS EKSTERNAL + REGEX LANTAI)
# ==========================================
import requests
import pandas as pd
import re

print("⏳ Mengunduh Kamus Bahasa Gaul (Slang Dictionary)...")
url_slang = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
try:
    df_slang = pd.read_csv(url_slang)
    slang_dict_external = dict(zip(df_slang['slang'], df_slang['formal']))
    print(f"✅ Berhasil memuat {len(slang_dict_external)} kata slang/gaul!")
except Exception as e:
    print(f"⚠️ Gagal mengunduh kamus slang. Error: {e}")
    slang_dict_external = {} 

def super_clean_text(text):
    # 1. Lowercase
    text = text.lower()
    
    # 2. POLESAN BARU: Menghapus kata yang berulang berurutan (Typo Stuttering)
    # Menangkap kata seperti "gedung gedung a" -> "gedung a", "sangat sangat" -> "sangat"
    text = re.sub(r'\b(\w+)(?:\s+\1\b)+', r'\1', text)
    
    # 3. Binding Lokasi Gedung
    text = re.sub(r'\b(gedung|gdng|gd\.?|tower|twr|blok|blk|lobby|lobi|area)\s*([a-z0-9]+)\b', r'gedung_\2', text)
    
    # 4. Binding Lokasi Lantai 
    text = re.sub(r'\b(lantai|lt\.?|level)\s*([a-z0-9]+)\b', r'lantai_\2', text)
    
    # 5. Binding Zona/Ruang
    text = re.sub(r'\b(ruang|rg\.?|kamar|kmr)\s*([a-z0-9]+)\b', r'ruang_\2', text)
    
    # Buang karakter aneh, sisakan huruf, angka, dan underscore (_) dari regex di atas
    text = re.sub(r'[^a-z0-9_]', ' ', text).strip()
    
    # Normalisasi bahasa gaul
    kata_kata = text.split()
    kata_normal = [slang_dict_external.get(kata, kata) for kata in kata_kata]
    text = " ".join(kata_normal)
    return text

print("⏳ Memulai proses Preprocessing...")
# PENTING: Terapkan pada data Training Augmentasi dan Data Testing murni
X_train_clean = X_train_aug.apply(super_clean_text)
X_test_clean = X_test_raw.apply(super_clean_text)

test_text = "Tolong dicek dong di gedung gedung A, lantai lantai 3 kotor banget air air netes"
print(f"Sebelum : {test_text}")
print(f"Sesudah : {super_clean_text(test_text)}")

print("✅ Cell 3 Selesai!")

⏳ Mengunduh Kamus Bahasa Gaul (Slang Dictionary)...
✅ Berhasil memuat 4331 kata slang/gaul!
⏳ Memulai proses Preprocessing...
Sebelum : Tolong dicek dong di gedung gedung A, lantai lantai 3 kotor banget air air netes
Sesudah : tolong dicek dong di gedung_a lantai_3 kotor banget air menetes
✅ Cell 3 Selesai!


In [ ]:
# Jalankan ini sebelum pipeline.fit
print("🔍 AUDIT DATA SEBELUM TRAINING:")
print(f"Tipe data X_train_clean: {type(X_train_clean)}")
print(f"Tipe data y_train_aug: {type(y_train_aug)}")

# Cek apakah ada nilai NaN atau None yang terselip
print(f"Jumlah NaN di X_train: {X_train_clean.isna().sum()}")
print(f"Contoh data y_train_aug (3 baris terakhir):")
print(y_train_aug.tail(3))

🔍 AUDIT DATA SEBELUM TRAINING:
Tipe data X_train_clean: <class 'pandas.core.series.Series'>
Tipe data y_train_aug: <class 'pandas.core.frame.DataFrame'>
Jumlah NaN di X_train: 0
Contoh data y_train_aug (3 baris terakhir):
     tipe_aset lokasi_gedung lokasi_lantai lokasi_zona  \
6093  APAR CO2  Gedung Utama            10      Tengah   
6094  APAR CO2  Gedung Utama            10      Tengah   
6095  APAR CO2  Gedung Utama            10      Tengah   

                 kategori_aset severity  
6093  Sistem Pemadam Kebakaran    Berat  
6094  Sistem Pemadam Kebakaran    Berat  
6095  Sistem Pemadam Kebakaran    Berat  


In [ ]:
# ==========================================
# CELL 4: TRAINING & MLFLOW TRACKING (TF-IDF ULTIMATE)
# ==========================================
import dagshub
import mlflow
import mlflow.sklearn
import numpy as np
from tqdm.notebook import tqdm  # Import pembuat visual bar untuk Jupyter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Inisialisasi DagsHub
dagshub.init(repo_owner='NazeeraAlthea', repo_name='exigen-smart-maintenance', mlflow=True)

# Ganti nama run agar mudah dibedakan di Dashboard DagsHub
with mlflow.start_run(run_name="Eksperimen_Ultimate_Regex_Bigram"):
    
    # --- HYPERPARAMETER SETUP ---
    MAX_FEAT = 3000
    N_GRAMS = (1, 2)  
    N_TREES = 300
    
    print("⏳ 1/2 Ekstraksi Fitur Teks (TF-IDF)...")
    # TF-IDF dieksekusi di luar loop agar matriks tidak dihitung ulang berkali-kali
    tfidf = TfidfVectorizer(max_features=MAX_FEAT, ngram_range=N_GRAMS, min_df=2, max_df=0.9)
    X_train_tfidf = tfidf.fit_transform(X_train_clean)
    
    print("\n🤖 2/2 Mulai melatih 6 Target AI (Perhatikan Progress Bar di bawah)...")
    
    # Siapkan kerangka kosong untuk MultiOutputClassifier
    clf = MultiOutputClassifier(RandomForestClassifier())
    estimators_ = []
    
    # --- IMPLEMENTASI TQDM VISUAL BAR ---
    # Loop berjalan 6 kali untuk masing-masing kolom (Tipe Aset, Gedung, Lantai, Zona, dll)
    for i in tqdm(range(y_train_aug.shape[1]), desc="Progress Training Target", unit="kolom"):
        rf_model = RandomForestClassifier(
            n_estimators=N_TREES, 
            class_weight='balanced', 
            n_jobs=-1,               # Tetap pakai semua core CPU
            random_state=42
            # Parameter verbose=2 DIHAPUS agar tampilan bar tidak rusak
        )
        # Latih dan simpan model ke dalam list
        estimators_.append(rf_model.fit(X_train_tfidf, y_train_aug.iloc[:, i]))
        
    # Masukkan kembali model-model yang sudah dilatih ke dalam kerangka MultiOutput
    clf.estimators_ = estimators_
    clf.classes_ = [estimator.classes_ for estimator in estimators_] 
    
    # 2. Bangun kembali Pipeline secara utuh agar fungsi predict() berjalan normal
    pipeline = Pipeline([
        ('tfidf', tfidf),
        ('clf', clf)
    ])
    
    print("\n📊 Melakukan prediksi & evaluasi pada data test murni...")
    y_pred = pipeline.predict(X_test_clean) 
    
    # 4. Evaluasi Exact Match Ratio
    exact_match = np.all(y_pred == y_test.values, axis=1).mean()
    print(f"\n🎯 Exact Match Ratio (Akurasi Penuh): {exact_match:.2%}")
    mlflow.log_metric("exact_match_ratio", exact_match)
    
    # 5. Evaluasi Akurasi Per Kolom
    print("-" * 30)
    for i, col in enumerate(kolom_target):
        acc = accuracy_score(y_test.iloc[:, i], y_pred[:, i])
        print(f"🔸 Akurasi {col.upper()}: {acc:.2%}")
        mlflow.log_metric(f"accuracy_{col}", acc)
        
    # 6. Catat Parameter Penting ke MLflow 
    mlflow.log_param("Feature_Extractor", f"TF-IDF {N_GRAMS} + Regex Location Binding")
    mlflow.log_param("Augmentation", "Back-Translation") 
    mlflow.log_param("Model", "Random Forest (Balanced Weights)")
    mlflow.log_param("max_features", MAX_FEAT)
    mlflow.log_param("n_estimators", N_TREES)
    
    # 7. Simpan model utuh
    mlflow.sklearn.log_model(pipeline, "model_tfidf_rf")
    
    print("\n✅ Cell 4 Selesai: Hasil Eksperimen TF-IDF berhasil dikirim ke DagsHub!")

Initialized MLflow to track repo "NazeeraAlthea/exigen-smart-maintenance"

Repository NazeeraAlthea/exigen-smart-maintenance initialized!

⏳ 1/2 Ekstraksi Fitur Teks (TF-IDF)...

🤖 2/2 Mulai melatih 6 Target AI (Perhatikan Progress Bar di bawah)...


Progress Training Target:   0%|          | 0/6 [00:00<?, ?kolom/s]


📊 Melakukan prediksi & evaluasi pada data test murni...

🎯 Exact Match Ratio (Akurasi Penuh): 81.48%
------------------------------
🔸 Akurasi TIPE_ASET: 97.35%
🔸 Akurasi LOKASI_GEDUNG: 93.65%
🔸 Akurasi LOKASI_LANTAI: 95.24%
🔸 Akurasi LOKASI_ZONA: 97.35%
🔸 Akurasi KATEGORI_ASET: 97.35%
🔸 Akurasi SEVERITY: 92.59%


2026/05/23 01:54:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 01:54:41 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



✅ Cell 4 Selesai: Hasil Eksperimen TF-IDF berhasil dikirim ke DagsHub!
🏃 View run Eksperimen_Ultimate_Regex_Bigram at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0/runs/0acc65d95c154ba49c0d00a4ddf40dc9
🧪 View experiment at: https://dagshub.com/NazeeraAlthea/exigen-smart-maintenance.mlflow/#/experiments/0


In [ ]:
import joblib

# Menentukan nama file untuk model Anda
model_filename = "../../../models/ticketing/ticket_v1.1.0.0_tfidf.pkl"

# Menyimpan seluruh Pipeline (Sastrawi + TF-IDF + Random Forest)
joblib.dump(pipeline, model_filename)

print(f"💾 Model AI Exigen berhasil disimpan secara lokal: {model_filename}")

# (Opsional) Jika Anda tetap ingin mencatat file ini ke MLflow DagsHub 
# tanpa membuat prosesnya macet/freeze, gunakan log_artifact:
try:
    mlflow.log_artifact(model_filename)
    print("✅ File model berhasil dikirim ke dashboard DagsHub sebagai artifact!")
except Exception as e:
    print(f"⚠️ Gagal mengirim ke DagsHub (Abaikan jika internet sedang tidak stabil): {e}")

💾 Model AI Exigen berhasil disimpan secara lokal: ../../models/ticketing/model_tfidf_rf_exigen.pkl
✅ File model berhasil dikirim ke dashboard DagsHub sebagai artifact!


In [ ]:
# ==========================================
# CELL 5: BLIND TESTING (AUTO-RANDOM DARI BANK SOAL)
# ==========================================
import random

print("=== 🧪 MEMULAI BLIND TESTING (OUT-OF-DISTRIBUTION) ===\n")

# 1. Bank Soal Ujian (Berbagai variasi aset, lokasi, dan gaya bahasa)
bank_keluhan = [
    "Tolonggg dong ini ac di ruang rapat lt 3 netes airnya parah bgt ngerusak karpet!! cepet di benerin ya mas",
    "Waduh lift penumpang lantai 5 tiba tiba nyangkut di tengah jalan trus lampunya mati. ada orang di dalem woy!",
    "Lantai granit depan lobby utama kelihatan kusam dan warnanya agak pudar, kayaknya perlu dipoles ulang deh",
    "Mas, ini lampu downlight di koridor lantai 2 gedung C kedap-kedip terus kayak di diskotik, bikin pusing.",
    "Gawat pak, panel listrik di ruang server lantai 1 ngeluarin asep dan bau gosong! Tolong segera dimatikan!",
    "Kran wastafel di toilet cowok lantai 4 lepas, airnya nyemprot kemana-mana ga bisa ditutup!",
    "CCTV di area parkir zona timur gambarnya burem banget dan kadang hilang sinyalnya.",
    "Mesin pompa air di basement suaranya kasar banget kaya ada besi gesekan, takut meledak nih.",
    "Pintu kaca otomatis di lobby gedung utama sensornya ngaco, kadang nutup sendiri pas ada orang lewat.",
    "Exhaust fan di dapur lantai 1 mati, asep masakannya jadi ngebul penuhin ruangan.",
    "Mas tolong cek mesin genset di gedung belakang, suaranya aneh banget ga kaya biasanya.",
    "Plafon di atas meja saya di lantai 3 ada rembesan air, takutnya tiba-tiba jebol.",
    "Tombol lift di lantai dasar rusak nih, dipencet berkali-kali ga nyala lampunya.",
    "Kabel LAN di meja saya lantai 2 zona tengah digigit tikus putus, ga bisa internetan ini.",
    "AC split di ruang HRD kurang dingin, cuma keluar angin doang."
]

# Mengambil 3 keluhan secara acak dari bank soal
keluhan_baru = random.sample(bank_keluhan, 3)

# 2. Preprocessing teks baru menggunakan fungsi yang sama saat training
print("⏳ Sedang memproses teks baru menggunakan Sastrawi dan RegEx...")
keluhan_clean = [super_clean_text(teks) for teks in keluhan_baru]

# 3. Prediksi menggunakan Pipeline
print("🤖 Meminta Model Random Forest untuk memprediksi...\n")
hasil_prediksi = pipeline.predict(keluhan_clean)

# 4. Menampilkan Hasil
for i, teks_asli in enumerate(keluhan_baru):
    print(f"📝 KELUHAN AWAM (WA) #{i+1}:")
    print(f"\"{teks_asli}\"")
    print("-" * 30)
    print("🎯 TEBAKAN AI FASE 1 (TAMPIL DI DASHBOARD):")
    
    print(f" 🔹 Tipe Aset     : {hasil_prediksi[i][0]}")
    print(f" 🔹 Gedung        : {hasil_prediksi[i][1]}")
    print(f" 🔹 Lantai        : {hasil_prediksi[i][2]}")
    print(f" 🔹 Zona          : {hasil_prediksi[i][3]}")
    print(f" 🔹 Kategori Dept : {hasil_prediksi[i][4]}")
    print(f" 🔹 Severity      : {hasil_prediksi[i][5]}")
    print("=" * 50 + "\n")

=== 🧪 MEMULAI BLIND TESTING (OUT-OF-DISTRIBUTION) ===

⏳ Sedang memproses teks baru menggunakan Sastrawi dan RegEx...
🤖 Meminta Model Random Forest untuk memprediksi...

📝 KELUHAN AWAM (WA) #1:
"Exhaust fan di dapur lantai 1 mati, asep masakannya jadi ngebul penuhin ruangan."
------------------------------
🎯 TEBAKAN AI FASE 1 (TAMPIL DI DASHBOARD):
 🔹 Tipe Aset     : Exhaust Fan
 🔹 Gedung        : Gedung Utama
 🔹 Lantai        : 1
 🔹 Zona          : Zona Tengah
 🔹 Kategori Dept : Ventilasi Sistem
 🔹 Severity      : Berat

📝 KELUHAN AWAM (WA) #2:
"AC split di ruang HRD kurang dingin, cuma keluar angin doang."
------------------------------
🎯 TEBAKAN AI FASE 1 (TAMPIL DI DASHBOARD):
 🔹 Tipe Aset     : AC Split
 🔹 Gedung        : Gedung Utama
 🔹 Lantai        : 1
 🔹 Zona          : Tengah
 🔹 Kategori Dept : Mechanical
 🔹 Severity      : Sedang

📝 KELUHAN AWAM (WA) #3:
"Mesin pompa air di basement suaranya kasar banget kaya ada besi gesekan, takut meledak nih."
----------------------------